# Repeatable Axolotl fine-tune

This guide runs the cookbook's [Axolotl fine-tune](https://github.com/nebius/serverless-ai-cookbook/tree/main/templates/job-axolotl-finetune) (QLoRA on `Qwen/Qwen2.5-0.5B`, spot H100) under `RepeatableJob` from [`repeatable_job.ipynb`](repeatable_job.ipynb). When the spot VM is taken away, a new attempt continues from the last checkpoint instead of starting over.

The cookbook command can't be rescheduled as-is:

| Cookbook template | Here |
|---|---|
| Checkpoints stay on the container disk and reach the bucket only after training succeeds | Each checkpoint is copied to the bucket as soon as it is saved |
| SIGTERM kills training; nothing is saved | SIGTERM saves a checkpoint at the current step, then exits 143 |
| `RUN_ID=$(date …)` changes every attempt | `CKPT_DIR` is fixed once per notebook run and shared by all attempts |
| `config.yaml` is downloaded with `curl` | `config.yaml` is uploaded with `inject_files` |
| 30 steps, no checkpoints | 3000 steps, a checkpoint every 200 |

Three files in [`axolotl/`](axolotl/) do this:

- [`resumable_axolotl.py`](axolotl/resumable_axolotl.py), the job's main process: copies the latest checkpoint from the bucket, runs `axolotl train`, forwards SIGTERM to it, and copies the final adapter to the bucket when training is done.
- [`checkpoint_plugin.py`](axolotl/checkpoint_plugin.py), an Axolotl plugin: copies each checkpoint to the bucket, and on SIGTERM tells the Trainer to save one more at the end of the current step.
- [`config.yaml`](axolotl/config.yaml), the cookbook config plus `save_steps`, `auto_resume_from_checkpoints`, and the plugin.

`RepeatableJob` lives in [`repeatable.py`](repeatable.py). Its loop is plain asyncio, so it runs wherever you await it: here in the notebook (section 5), in an orchestrator such as Prefect, or in an optional CPU job on Nebius, the supervisor from [`supervisor.py`](supervisor.py) (section 6), which keeps relaunching after the notebook is closed. Every attempt, state change, and reschedule decision is logged.

Preemption is simulated by stopping the job's VM, which is what a spot preemption does.

**Before you start**

- `uv sync` in the repo root, then start Jupyter from this directory, for example `uv run --with jupyter jupyter lab examples/signals`.
- A Nebius CLI profile (`nebius profile create`) and a bucket you can mount read-write. Set `BUCKET_ID` below to its id (`storagebucket-…`).
- For section 6: a service account in the `editors` group with an authorized key, and a MysteryBox secret holding the key's private PEM. A job's container gets no Nebius credentials of its own, so the supervisor signs in as this account.
- The job runs on a spot H100 (`gpu-h100-sxm`, one GPU). **Sections 5 and 6 each start three billable GPU attempts; run one of them.** Section 6 also runs a CPU job for as long as the loop lasts.

In [ ]:
import nebius_serverless as ns

ns.diagnostics()

In [ ]:
import asyncio
import logging
from datetime import datetime, timezone
from pathlib import Path

from repeatable import RepeatableJob, end_reason, instance_id
from supervisor import ServiceAccount

# Show the loop's log lines (attempts, states, decisions) in the notebook.
logger = logging.getLogger("repeatable")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S")
    )
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

# Bucket id, not name: nebius storage bucket get-by-name --name <bucket>
BUCKET_ID = "storagebucket-e008847641561172892054"  # ez-bucket-1
# One name for the run: attempt jobs, the supervisor, the MLflow run, and
# the checkpoint folder.
# Launch time in UTC first (z = UTC), so runs sort by when they started:
# 20261118-130055z-qwen05b-qlora-1, ...-2, ...-supervisor
RUN_NAME = f"{datetime.now(timezone.utc):%Y%m%d-%H%M%S}z-qwen05b-qlora"
CKPT_DIR = f"/ckpt/repeatable-axolotl/{RUN_NAME}"
IMAGE = "docker.io/axolotlai/axolotl:main-20260309-py3.11-cu128-2.9.1"
PREEMPT_AFTER = 300  # simulated preemption: stop the VM after this many s
FILES = Path("axolotl").resolve()  # uploaded into each attempt

# MLflow server (nebius msp mlflow v1alpha1 cluster list) and its login
MLFLOW_URI = (
    "https://public-tracking-e00-th0rycpzb30p26v-ebn5j97mr4nqat7"
    "-mlflow.gw.msp.eu-north1.nebius.cloud"
)  # mlflow1
MLFLOW_USER = "admin"
# Local file with MLFLOW_TRACKING_PASSWORD=..., readable only by you
SECRETS_ENV = Path.home() / ".nebius" / "mlflow1.env"

RUN_NAME

## 1. Why did an attempt end?

`end_reason` maps the final state to a reason, as in [`repeatable_job.ipynb`](repeatable_job.ipynb#1.-Why-did-an-attempt-end?). Stopping the VM ends the attempt `ERROR` with `VMStopped`, which counts as `platform` and is rescheduled. Running out of spot capacity (`NotEnoughResources`) is also `platform`. A bug in the config ends `FAILED` (`job`) and is not rescheduled.

In [ ]:
from repeatable import REASONS

REASONS

## 2. The wrapper

[`repeatable.py`](repeatable.py) holds `RepeatableJob`, the same loop as in [`repeatable_job.ipynb`](repeatable_job.ipynb#2.-The-wrapper): submit the job, wait for it to end, submit again after a `cancelled` or `platform` end, stop after `completed` or `job`. To stop for good, call `cancel_async()`.

It logs every step to the `repeatable` logger: submission, each state change of the attempt (with the VM it runs on), how it ended and after how long, and whether it is rescheduled. `name` names everything: attempt `n` is created as job `{name}-{n}` with `REPEATABLE_NAME` and `REPEATABLE_ATTEMPT` in its environment, the supervisor is `{name}-supervisor`, and the training plugin names the MLflow run `{name}`. `repeatable.attempts` keeps the same history as `Attempt` records, and `repeatable.summary()` prints one line per attempt.

Where the loop runs is up to you:

- **Here**, `await repeatable.run_async()` (section 5). Interrupting the notebook stops the loop.
- **In an orchestrator**, for example a Prefect flow. Pass its logger so the lines land in the flow run:

  ```python
  from prefect import flow, get_run_logger

  @flow
  async def finetune():
      repeatable = RepeatableJob(job, max_attempts=5, logger=get_run_logger())
      await repeatable.run_async()
  ```

- **In a CPU job on Nebius**, `await repeatable.launch_async(account)` (section 6), when nothing else stays up for the whole run. This uses the optional [`supervisor.py`](supervisor.py).

## 3. Describe the job

The machine settings come from the cookbook template: the Axolotl image, a spot H100, a 500 GiB disk. The SDK already sets the template's 16 GiB `/dev/shm`.

- `volumes` mounts the bucket by id at `/ckpt`. Checkpoints and the final adapter go to `CKPT_DIR` there.
- `inject_files` uploads the wrapper, the plugin, and the config to `/app`.
- The bucket mount accepts file contents but not permission or timestamp changes, so `shutil.copytree` and `copy2` fail there. The scripts copy with `shutil.copyfile`.
- MLflow: `config.yaml` sets `mlflow_experiment_name`, and the job gets the server's URL and user as `MLFLOW_TRACKING_*` variables. The password comes from the local file `SECRETS_ENV`, uploaded as `/app/secrets.env`: injected files are stored encrypted and their contents don't show in the job's record, while `env` values do. (A MysteryBox secret works too, but the account that creates the job, here the supervisor's service account, needs `mysterybox.payload-viewer` on it.) All attempts log to one run: the first saves its id next to the checkpoints, later ones reattach with `MLFLOW_RUN_ID` and skip logging parameters (MLflow rejects a parameter whose value changes within a run). Loss curves continue at the resumed step, the tag `repeatable.attempt` shows the latest attempt, and the final adapter is logged as artifacts. The wrapper installs `mlflow-skinny` if the image lacks it.
",
    "- `PYTHONPATH=/app` lets Axolotl import `checkpoint_plugin` from `plugins:` in the config.

The model and dataset are downloaded again in every attempt. That takes about a minute for a 0.5B model; for larger models, cache them in the bucket.

In [ ]:
job = ns.Job(
    ["python", "-u", "/app/resumable_axolotl.py"],
    image=IMAGE,
    gpu="h100",
    pricing="spot",
    disk="500Gi",
    timeout="2h",
    volumes=[f"{BUCKET_ID}:/ckpt:rw"],
    inject_files=[
        f"{FILES / name}:/app/{name}"
        for name in (
            "resumable_axolotl.py",
            "checkpoint_plugin.py",
            "config.yaml",
        )
    ]
    + [f"{SECRETS_ENV}:/app/secrets.env"],
    env={
        "CKPT_DIR": CKPT_DIR,
        "PYTHONPATH": "/app",
        "MLFLOW_TRACKING_URI": MLFLOW_URI,
        "MLFLOW_TRACKING_USERNAME": MLFLOW_USER,
    },
    name="repeatable-axolotl",
)

## 4. Simulate preemption

`simulate_preemption` stops the VM under each of the first `times` attempts, `after` seconds after the attempt starts running. It finds the VM in `status.instances[0].compute_instance_id` and calls `nebius compute instance stop`. `attempts` returns the attempt ids so far: from `repeatable.attempts` when the loop runs here, from the supervisor's logs when it runs in the cloud.

The container gets SIGTERM about 4 seconds after the stop, and SIGKILL about 9.5 seconds after that. In that window the plugin finishes the current step, saves `checkpoint-N`, copies it to the bucket, and exits. If the copy doesn't finish in time, the next attempt resumes from the previous checkpoint, at most `save_steps` (200) steps earlier.

`after` counts from `RUNNING`, which includes the model download and dataset preparation, so it must be long enough for training to have started.

In [ ]:
import re

WAITING = ("PROVISIONING", "STARTING", "IMAGE_PULLING")
# `nebius ai job logs` returns the last 100 lines by default; 1000 is the max.
FULL_LOG = ("--since", "24h", "--tail", "1000")


async def nebius(*args: str) -> str:
    """Run the nebius CLI and return its stdout."""
    proc = await asyncio.create_subprocess_exec(
        "nebius", *args, stdout=asyncio.subprocess.PIPE
    )
    out, _ = await proc.communicate()
    if proc.returncode != 0:
        raise RuntimeError(f"nebius {' '.join(args)} failed")
    return out.decode()


async def logged_attempts(supervisor_id: str) -> list[str]:
    """Attempt ids from the supervisor's `attempt N/M: created <id>` lines."""
    logs = await nebius("ai", "job", "logs", supervisor_id, *FULL_LOG)
    return re.findall(r"attempt \d+/\d+: created (\S+)", logs)


async def simulate_preemption(attempts, *, after: float, times: int) -> None:
    """Stop the VM of the first `times` attempts, `after` s into RUNNING."""
    for n in range(times):
        while len(ids := await attempts()) <= n:  # wait for attempt n + 1
            await asyncio.sleep(10)
        probe = await ns.jobs.get_async(ids[n], quiet=True)
        while probe.status in WAITING:
            await asyncio.sleep(5)
            await probe.get_async(quiet=True)
        if probe.status == "RUNNING":
            await asyncio.sleep(after)
            await probe.get_async(quiet=True)
            vm = instance_id(probe)
            await nebius("compute", "instance", "stop", "--id", vm)
            print(f"simulated preemption: stopped {vm} under {probe.id}")

## 5. Option A: run the loop here

Expect three attempts: the first two end `ERROR` `VMStopped` and are rescheduled, the third completes. Each attempt waits for a spot H100 and pulls the Axolotl image, which can take several minutes.

Interrupting this cell stops the loop but not the current attempt. Cancel it with `await repeatable.cancel_async()`.

In [ ]:
repeatable = RepeatableJob(job, name=RUN_NAME, max_attempts=5)


async def local_attempts() -> list[str]:
    return [a.job_id for a in repeatable.attempts]


run, _ = await asyncio.gather(
    repeatable.run_async(),
    simulate_preemption(local_attempts, after=PREEMPT_AFTER, times=2),
)
attempt_ids = await local_attempts()
print(repeatable.summary())

## 6. Option B: run the loop in a CPU job

`launch_async` starts the supervisor from [`supervisor.py`](supervisor.py) and returns at once. The supervisor logs the same lines as section 5, ending with the attempt summary. The supervisor runs on the default CPU preset with on-demand pricing, so it isn't preempted itself. Its `timeout` bounds the whole loop.

The service account needs an authorized key. Create one and store the private key in MysteryBox, for example:

```bash
SA=serviceaccount-...      # the account
PROJECT=project-...        # your project
nebius iam auth-public-key generate --service-account-id $SA \
  --output-format pem --output sa.pem
nebius iam auth-public-key list --parent-id $PROJECT  # newest key of $SA: public_key_id
nebius mysterybox secret create --parent-id $PROJECT \
  --name repeatable-supervisor-key \
  --secret-version-payload "[{\"key\": \"NEBIUS_SA_PRIVATE_KEY\", \"string_value\": $(jq -Rs . sa.pem)}]"
rm sa.pem
```

In [ ]:
ACCOUNT = ServiceAccount(
    id="serviceaccount-e00pe5phdy0w0wxd23",  # perfect-pipeline
    public_key_id="publickey-e00rr9c847fm7w20dr",  # authorized key
    private_key_secret="mbsec-e00nv8q43vwjefc8mm",  # its private PEM
)

Stop the VMs from here as in option A; the attempt ids come from the supervisor's logs. This cell waits for the supervisor to finish, but closing the notebook doesn't stop it. To stop for good, cancel the supervisor: `await supervisor.cancel_async()`.

In [ ]:
repeatable = RepeatableJob(job, name=RUN_NAME, max_attempts=5)
supervisor = await repeatable.launch_async(ACCOUNT, timeout="12h")


async def cloud_attempts() -> list[str]:
    return await logged_attempts(supervisor.id)


await asyncio.gather(
    supervisor.wait_async(),
    simulate_preemption(cloud_attempts, after=PREEMPT_AFTER, times=2),
)
attempt_ids = await cloud_attempts()
print(supervisor.status)
!nebius ai job logs {supervisor.id} --since 24h --tail 1000

## 7. Check the attempts

Each attempt should resume from the checkpoint the previous one saved on SIGTERM. The last attempt copies the adapter to `CKPT_DIR/final`.

In [ ]:
PATTERN = "starting fresh|resuming|SIGTERM|copied|training done"
for n, job_id in enumerate(attempt_ids, start=1):
    r = await ns.jobs.get_async(job_id, quiet=True)
    reason, code = end_reason(r)
    print(f"attempt {n}: {r.id} {r.status} {code}".rstrip(), f"({reason})")
    !nebius ai job logs {r.id} --since 24h --tail 1000 | grep -E "{PATTERN}"

## 8. Download the adapter

The adapter is in the bucket under `CKPT_DIR/final` (without the `/ckpt` prefix). For example, with the AWS CLI configured for Nebius Object Storage:

```bash
aws s3 cp --recursive s3://<bucket>/repeatable-axolotl/<run>/final ./adapter
```